# Advanced: Multi-Strategy Cell Suppression with PAMOLA.CORE

**Goal**: Master all 7 cell suppression strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Mean-based with IQR outlier detection
- **Strategy 2**: Group-mean with rare value detection
- **Strategy 3**: Constant with Z-score outlier detection
- Calculate advanced privacy metrics (k-anonymity)
- Analyze information loss across strategies
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/suppression/
        └── 02_cell_suppression_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.suppression.cell_op import CellSuppressionOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record patient dataset
- Auto-creates sample if file not found
- Review data structure and identify outliers

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics:
  - Numeric fields with ranges (age, temperature, blood pressure, heart rate)
  - Categorical fields (diagnosis, risk_group)
  - Outliers in temperature (95-105°F vs normal 97-100°F)

**Note:** Generated data includes realistic outliers and rare values for suppression demonstration

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate base patient data
    ages = np.random.randint(18, 85, n_records)
    
    # Normal temperature readings (98.6°F ± 1°F) with outliers
    temperatures = np.random.normal(98.6, 1.0, n_records - 50)
    outlier_temps_low = np.random.uniform(95.0, 96.5, 25)  # Low outliers
    outlier_temps_high = np.random.uniform(103.0, 105.5, 25)  # High outliers
    all_temps = np.concatenate([temperatures, outlier_temps_low, outlier_temps_high])
    np.random.shuffle(all_temps)
    
    # Blood pressure with outliers
    bp_systolic = np.random.randint(110, 135, n_records - 30)
    bp_systolic_outliers = np.random.choice([90, 95, 180, 190], 30)  # Outliers
    all_bp_systolic = np.concatenate([bp_systolic, bp_systolic_outliers])
    np.random.shuffle(all_bp_systolic)
    
    bp_diastolic = np.random.randint(70, 85, n_records)
    
    # Heart rate with outliers
    heart_rate = np.random.randint(65, 95, n_records - 40)
    hr_outliers = np.random.choice([45, 50, 120, 130, 140], 40)  # Outliers
    all_heart_rate = np.concatenate([heart_rate, hr_outliers])
    np.random.shuffle(all_heart_rate)
    
    # Diagnosis with rare values
    common_diagnoses = ['Healthy', 'Flu', 'Cold', 'Hypertension', 'Infection']
    rare_diagnoses = ['Rare_Disease_A', 'Rare_Disease_B']
    diagnoses = (
        list(np.random.choice(common_diagnoses, n_records - 15)) +
        list(np.random.choice(rare_diagnoses, 15))
    )
    np.random.shuffle(diagnoses)
    
    # Risk groups for group-based strategies
    risk_groups = np.random.choice(['Low', 'Medium', 'High', 'Critical'], n_records)
    
    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, n_records + 1)],
        'age': ages,
        'temperature': all_temps,
        'blood_pressure_systolic': all_bp_systolic,
        'blood_pressure_diastolic': bp_diastolic,
        'heart_rate': all_heart_rate,
        'diagnosis': diagnoses,
        'risk_group': risk_groups
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:30s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:30s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `strategy1_field`: For mean-based outlier suppression
   - `strategy2_field`: For group-mean rare value suppression
   - `strategy3_field`: For constant Z-score suppression
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Statistics per field (min, max, mean, std)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource created (✓)

**Note:** All fields must exist in dataset and be numeric for suppression strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "current_salary_cad",  # Mean-based with IQR outliers
    "strategy2_field": "current_salary_cad",  # Group-mean with rare values
    "strategy3_field": "current_salary_cad"   # Constant with Z-score outliers
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    # Validate numeric type
    if not pd.api.types.is_numeric_dtype(df[field_name]):
        raise ValueError(
            f"❌ Field '{field_name}' must be numeric for cell suppression!\n"
            f"Current type: {df[field_name].dtype}"
        )
    
    # Display statistics
    print(f"  ✓ {strategy:20s}: '{field_name}'")
    print(f"      Min:  {df[field_name].min():.2f}")
    print(f"      Max:  {df[field_name].max():.2f}")
    print(f"      Mean: {df[field_name].mean():.2f}")
    print(f"      Std:  {df[field_name].std():.2f}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_cell_001",
    task_type="multi_strategy",
    description="Multi-strategy cell suppression",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Mean-Based with IQR Outlier Detection

**How to use:**
- Detects outliers using IQR method (1.5 × IQR)
- Replaces outliers with field mean
- Run to execute strategy

**Key parameters:**
- `suppression_strategy='mean'`: Replace with mean value
- `suppress_if='outlier'`: Trigger on outlier detection
- `outlier_method='iqr'`: Use Interquartile Range method
- `outlier_threshold=1.5`: Standard IQR multiplier
- `mode='ENRICH'`: Adds new column, keeps original

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → detect → suppress → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Outlier count and suppression statistics

**Note:** IQR method is robust to extreme outliers. Best for normally distributed data

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: MEAN-BASED WITH IQR OUTLIER DETECTION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Mean+IQR",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = CellSuppressionOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    
    # Suppression strategy
    suppression_strategy='mean',
    
    # Automatic suppression trigger
    suppress_if='outlier',
    outlier_method='iqr',
    outlier_threshold=1.5,
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_mean_iqr",
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_mean_iqr',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_mean_iqr' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_mean_iqr"
    
    suppressed_mask_s1 = df[field_s1] != result_df_s1[output_field_s1]
    print(f"\n📊 Outliers suppressed: {suppressed_mask_s1.sum()} / {len(df)} cells ({(suppressed_mask_s1.sum()/len(df)*100):.1f}%)")
    print(f"   Original range: [{df[field_s1].min():.2f}, {df[field_s1].max():.2f}]")
    print(f"   After suppression: [{result_df_s1[output_field_s1].min():.2f}, {result_df_s1[output_field_s1].max():.2f}]")

## STRATEGY 2: Group-Mean with Rare Value Detection

**How to use:**
- Groups data by field
- Replaces with group-specific mean
- Detects and suppresses rare values
- Run to execute strategy

**Key parameters:**
- `suppression_strategy='group_mean'`: Group-based mean replacement
- `group_by_field=group_by_field`: Group by field
- `min_group_size=5`: Minimum records per group
- `suppress_if='rare'`: Detect rare values
- `rare_threshold=10`: Values appearing <10 times are rare

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → group stats → suppress → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Group statistics and rare value counts

**Note:** Group-based strategies preserve group-level patterns while protecting outliers

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: GROUP-MEAN WITH RARE VALUE DETECTION")
print("=" * 80 + "\n")

group_by_field = 'location_city'

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Group-Mean",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = CellSuppressionOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    
    # Group-based suppression
    suppression_strategy='group_mean',
    group_by_field=group_by_field,
    min_group_size=5,
    
    # Rare value detection
    suppress_if='rare',
    rare_threshold=10,
    
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_group_mean",
    output_format='csv',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_group_mean',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_group_mean' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_group_mean"
    
    suppressed_mask_s2 = df[field_s2] != result_df_s2[output_field_s2]
    print(f"\n📊 Rare values suppressed: {suppressed_mask_s2.sum()} / {len(df)} cells ({(suppressed_mask_s2.sum()/len(df)*100):.1f}%)")
    
    print(f"\n📋 Group Statistics:")

if group_by_field not in df.columns:
    print(f"   ⚠️  {group_by_field} not found – skipping group statistics")
else:
    for group in df[group_by_field].dropna().unique():
        group_mask = df[group_by_field] == group
        group_mean = df.loc[group_mask, field_s2].mean()
        group_count = int(group_mask.sum())
        print(f"   {str(group):10s}: n={group_count:3d}, mean={group_mean:.2f}")

## STRATEGY 3: Constant with Z-Score Outlier Detection

**How to use:**
- Detects outliers using Z-score method (3σ threshold)
- Replaces outliers with constant value
- More aggressive than IQR method
- Run to execute strategy

**Key parameters:**
- `suppression_strategy='constant'`: Replace with constant value
- `suppression_value=10`: Fixed replacement value
- `suppress_if='outlier'`: Trigger on outlier detection
- `outlier_method='zscore'`: Use Z-score method
- `outlier_threshold=3.0`: 3 standard deviations (99.7% rule)

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → detect → suppress → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Z-score distribution and suppression statistics

**Note:** Z-score method assumes normal distribution. Good for large datasets

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: CONSTANT WITH Z-SCORE OUTLIER DETECTION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Constant+Zscore",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = CellSuppressionOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    
    # Constant suppression
    suppression_strategy='constant',
    suppression_value=10,  # Normal heart rate
    
    # Z-score outlier detection
    suppress_if='outlier',
    outlier_method='zscore',
    outlier_threshold=3.0,  # 3 standard deviations
    
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_constant_zscore",
    output_format='csv',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_constant_zscore',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_constant_zscore' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_constant_zscore"
    
    suppressed_mask_s3 = df[field_s3] != result_df_s3[output_field_s3]
    print(f"\n📊 Outliers suppressed: {suppressed_mask_s3.sum()} / {len(df)} cells ({(suppressed_mask_s3.sum()/len(df)*100):.1f}%)")
    print(f"   Replacement value: {operation_s3.suppression_value}")
    print(f"   Original range: [{df[field_s3].min():.2f}, {df[field_s3].max():.2f}]")
    print(f"   After suppression: [{result_df_s3[output_field_s3].min():.2f}, {result_df_s3[output_field_s3].max():.2f}]")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times for all 3 strategies
- Compare suppression rates and effectiveness
- Identify best strategy for your use case

**What you'll see:**
- Execution time for each strategy (seconds)
- Suppression rate comparison (% cells modified)
- Total processing time for all strategies
- Range changes (before → after)

**Strategy selection guide:**
- **Mean+IQR**: Balanced approach, preserves distribution shape
- **Group-Mean**: Context-aware, preserves group patterns
- **Constant+Z-score**: Maximum standardization, simplest output

**Note:** Lower suppression rate + maintained distribution = better utility

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Mean+IQR):         {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Group-Mean):       {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Constant+Z-score): {elapsed_s3:6.2f}s")
print(f"  Total:                         {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📈 Suppression Rates:")
if 'suppressed_mask_s1' in locals():
    print(f"  Strategy 1: {suppressed_mask_s1.sum():4d} cells ({(suppressed_mask_s1.sum()/len(df)*100):5.2f}%)")
if 'suppressed_mask_s2' in locals():
    print(f"  Strategy 2: {suppressed_mask_s2.sum():4d} cells ({(suppressed_mask_s2.sum()/len(df)*100):5.2f}%)")
if 'suppressed_mask_s3' in locals():
    print(f"  Strategy 3: {suppressed_mask_s3.sum():4d} cells ({(suppressed_mask_s3.sum()/len(df)*100):5.2f}%)")

print(f"\n📊 Data Distribution Impact:")
print(f"  Strategy 1 ({FIELD_CONFIG['strategy1_field']}):")
print(f"    Before:  mean={df[FIELD_CONFIG['strategy1_field']].mean():.2f}, std={df[FIELD_CONFIG['strategy1_field']].std():.2f}")
if 'result_df_s1' in locals():
    print(f"    After:   mean={result_df_s1[output_field_s1].mean():.2f}, std={result_df_s1[output_field_s1].std():.2f}")

print(f"\n  Strategy 2 ({FIELD_CONFIG['strategy2_field']}):")
print(f"    Before:  mean={df[FIELD_CONFIG['strategy2_field']].mean():.2f}, std={df[FIELD_CONFIG['strategy2_field']].std():.2f}")
if 'result_df_s2' in locals():
    print(f"    After:   mean={result_df_s2[output_field_s2].mean():.2f}, std={result_df_s2[output_field_s2].std():.2f}")

print(f"\n  Strategy 3 ({FIELD_CONFIG['strategy3_field']}):")
print(f"    Before:  mean={df[FIELD_CONFIG['strategy3_field']].mean():.2f}, std={df[FIELD_CONFIG['strategy3_field']].std():.2f}")
if 'result_df_s3' in locals():
    print(f"    After:   mean={result_df_s3[output_field_s3].mean():.2f}, std={result_df_s3[output_field_s3].std():.2f}")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Suppression strategy, cells suppressed, rate, detection method
2. **Output Data**: Preview with original and suppressed columns
3. **Visualizations**: Before/after comparison charts (latest batch only)

**Validate:**
- ✅ Suppression rate 3-10% (only outliers/rare values affected)
- ✅ Distribution normalized (outliers removed)
- ✅ Group patterns preserved (Strategy 2)
- ✅ Constant replacement applied (Strategy 3)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_mean_iqr', 'Strategy 1: Mean + IQR Outlier Detection'),
    ('strategy2_group_mean', 'Strategy 2: Group-Mean + Rare Values'),
    ('strategy3_constant_zscore', 'Strategy 3: Constant + Z-Score Outliers')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                print(f"   Operation:        {metrics.get('operation_type', 'N/A')}")
                print(f"   Strategy:         {metrics.get('suppression_strategy', 'N/A')}")
                print(f"   Cells Suppressed: {metrics.get('cells_suppressed', 0):,}")
                print(f"   Suppression Rate: {metrics.get('suppression_rate', 0):.2f}%")
                print(f"   Duration:         {metrics.get('duration_seconds', 0):.4f}s")
                
                # Strategy-specific metrics
                if 'outlier_method' in metrics:
                    print(f"   Outlier Method:   {metrics.get('outlier_method', 'N/A')}")
                    print(f"   Outlier Threshold: {metrics.get('outlier_threshold', 0):.2f}")
                
                if 'group_count' in metrics:
                    print(f"   Groups:           {metrics.get('group_count', 0)}")
                    print(f"   Min Group Size:   {metrics.get('min_group_size', 0)}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Output Data Preview (NEWEST)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        output_files = sorted(
            list(output_dir.glob('*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if output_files:
            print(f"\n📄 OUTPUT: {output_files[0].name}")
            try:
                output_df = pd.read_csv(output_files[0])
                print(f"   Records: {len(output_df):,}")
                print(f"   Columns: {len(output_df.columns)}")
                print(f"\n   Preview (first 5 rows):")
                # Show only relevant columns
                relevant_cols = [col for col in output_df.columns if any(field in col for field in FIELD_CONFIG.values())]
                if relevant_cols:
                    print(output_df[relevant_cols].head().to_string(index=False))
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Calculate k-anonymity for original and processed data
- Compare privacy improvements across strategies
- Run AFTER all 3 strategies complete

**What you'll see:**
- Original data: min k, avg k, vulnerable group count
- After each strategy: improved k-values with multipliers (e.g., 3x)
- Final k-anonymity level (e.g., "7-anonymous")

**Privacy targets:**
- Min k ≥ 5: Basic privacy protection
- Min k ≥ 10: Strong privacy protection
- Zero vulnerable groups: Ideal state

**Note:** Cell suppression improves k-anonymity by removing outliers that uniquely identify individuals

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics"""
    # Filter to only existing columns
    existing_qis = [q for q in quasi_identifiers if q in df.columns]
    if not existing_qis:
        return None

    group_sizes = df.groupby(existing_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

# Check if FIELD_CONFIG exists (strategies completed)
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    
    # Use group_by_field as quasi-identifiers
    quasi_ids_original = [group_by_field]
    
    # Calculate for original data
    k_orig = calculate_k_anonymity(df, quasi_ids_original)
    print(f"📊 ORIGINAL DATA:")
    print(f"   Quasi-identifiers: {', '.join(quasi_ids_original)}")
    print(f"   Min k: {k_orig['min_k']}")
    print(f"   Avg k: {k_orig['avg_k']:.2f}")
    print(f"   Vulnerable groups: {k_orig['vulnerable_groups']}")
    
    # Calculate after Strategy 1
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        # Use same quasi-identifiers since we only modified numeric fields
        k_s1 = calculate_k_anonymity(result_df_s1, quasi_ids_original)
        
        print(f"\n✨ AFTER STRATEGY 1 (Mean + IQR):")
        print(f"   Min k: {k_s1['min_k']} ({k_s1['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_s1['avg_k']:.2f} ({k_s1['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_s1['vulnerable_groups']}")
    
    # Calculate after Strategy 2
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        k_s2 = calculate_k_anonymity(result_df_s2, quasi_ids_original)
        
        print(f"\n✨ AFTER STRATEGY 2 (Group-Mean):")
        print(f"   Min k: {k_s2['min_k']} ({k_s2['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_s2['avg_k']:.2f} ({k_s2['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_s2['vulnerable_groups']}")
    
    # Calculate after Strategy 3
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        k_s3 = calculate_k_anonymity(result_df_s3, quasi_ids_original)
        
        print(f"\n✨ AFTER STRATEGY 3 (Constant + Z-score):")
        print(f"   Min k: {k_s3['min_k']} ({k_s3['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_s3['avg_k']:.2f} ({k_s3['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_s3['vulnerable_groups']}")
        
        print(f"\n🎯 Final dataset is {k_s3['min_k']}-anonymous")
        
    if not any(var in locals() for var in ['result_df_s1', 'result_df_s2', 'result_df_s3']):
        print("\n⚠️  No strategy results available - run all strategies first!")
        
except NameError:
    print("⚠️  Run Step 3 (Setup Shared Environment) first!")

## Step 7: Export Final Data

**How to use:**
- Export final processed datasets from all strategies
- Each strategy gets its own output folder
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Available columns list
- Export confirmation (file path, row/column count)
- Preview of first 5 rows
- Suppression statistics (cells modified, rate)

**Output structure:**
```
advanced_output/
├── strategy1_mean_iqr/suppressed_data.csv
├── strategy2_group_mean/suppressed_data.csv
└── strategy3_constant_zscore/suppressed_data.csv
```

**Note:** Files include both original and suppressed columns for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
except NameError:
    print("❌ Run Step 3 first!")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Mean-Based with IQR
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: MEAN-BASED WITH IQR OUTLIER DETECTION")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_mean_iqr'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_mean_iqr"
    
    if output_col_s1 in result_df_s1.columns:
        # Export with original and suppressed columns
        export_cols_s1 = ['patient_id', field_s1, output_col_s1, 'age', group_by_field, 'diagnosis']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        # Save to CSV
        output_path_s1 = s1_dir / 'mean_iqr_suppressed_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        # Preview
        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s1.head())
        
        # Statistics
        suppressed = (result_df_s1[field_s1] != result_df_s1[output_col_s1]).sum()
        print(f"\n📈 Suppression Statistics:")
        print(f"   Cells suppressed: {suppressed:,}")
        print(f"   Suppression rate: {(suppressed/len(result_df_s1)*100):.2f}%")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Group-Mean
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: GROUP-MEAN WITH RARE VALUE DETECTION")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_group_mean'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_mean_iqr"
    output_col_s2 = f"{field_s2}_group_mean"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['patient_id', field_s1, output_col_s1,
                          field_s2, output_col_s2, group_by_field, 'diagnosis']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'group_mean_suppressed_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s2.head())
        
        suppressed = (result_df_s2[field_s2] != result_df_s2[output_col_s2]).sum()
        print(f"\n📈 Suppression Statistics:")
        print(f"   Cells suppressed: {suppressed:,}")
        print(f"   Suppression rate: {(suppressed/len(result_df_s2)*100):.2f}%")
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Constant with Z-score
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: CONSTANT WITH Z-SCORE OUTLIER DETECTION")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_constant_zscore'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_mean_iqr"
    output_col_s2 = f"{field_s2}_group_mean"
    output_col_s3 = f"{field_s3}_constant_zscore"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['patient_id', 
                          field_s1, output_col_s1,
                          field_s2, output_col_s2,
                          field_s3, output_col_s3,
                          group_by_field, 'diagnosis']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'constant_zscore_suppressed_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview (first 5 rows):")
        print(final_df_s3.head())
        
        suppressed = (result_df_s3[field_s3] != result_df_s3[output_col_s3]).sum()
        print(f"\n📈 Suppression Statistics:")
        print(f"   Cells suppressed: {suppressed:,}")
        print(f"   Suppression rate: {(suppressed/len(result_df_s3)*100):.2f}%")
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 suppression strategies implemented and compared
- ✅ Outlier detection using IQR and Z-score methods
- ✅ Group-based suppression with rare value handling
- ✅ Privacy metrics calculated (k-anonymity)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Cell suppression** provides fine-grained privacy protection at value level
- **IQR method** is more conservative, detects only extreme outliers
- **Z-score method** is more aggressive, assumes normal distribution
- **Group-based strategies** preserve context while protecting privacy
- Suppression rates typically 3-10% for well-designed strategies

**Strategy recommendations:**
- **Use Strategy 1 (Mean+IQR)** when:
  - Data distribution is skewed
  - Need conservative outlier detection
  - Want to preserve overall distribution shape
  
- **Use Strategy 2 (Group-Mean)** when:
  - Data has meaningful groups (risk levels, categories)
  - Need context-aware suppression
  - Want to detect rare values
  
- **Use Strategy 3 (Constant+Z-score)** when:
  - Data is normally distributed
  - Need maximum standardization
  - Prefer fixed replacement values

**Next steps:**
- Test with your own datasets
- Adjust thresholds for optimal privacy/utility balance
- Combine with other anonymization techniques
- Deploy to production environment

**Other Suppression Operations:**
- 📗 **Attribute Suppression**: Remove entire columns
- 📙 **Record Suppression**: Remove entire rows based on criteria

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)